In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist.gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_fat_pedido_total  = spark.table("olist.silver.fat_pedido_total")
df_fat_itens_pedidos = spark.table("olist.silver.fat_itens_pedidos")
df_dim_produtos      = spark.table("olist.silver.dim_produtos")
df_dim_categoria     = spark.table("olist.silver.dim_categoria_produtos_traducao")

In [0]:
# Enriquece itens com categoria do produto e tradução para inglês
df_itens_enriquecido = (
    df_fat_itens_pedidos
    .join(df_dim_produtos, "id_produto", "left")
    .join(
        df_dim_categoria,
        df_dim_produtos["categoria_produto"] == df_dim_categoria["nome_produto_pt"],
        "left"
    )
    .select(
        "id_pedido",
        "id_produto",
        "preco_BRL",
        F.coalesce(
            F.col("nome_produto_en"),
            F.col("categoria_produto")
        ).alias("categoria_produto")
    )
)

# Junta com fat_pedido_total para obter data e cotação do dólar
df_vendas = (
    df_fat_pedido_total
    .join(df_itens_enriquecido, "id_pedido", "inner")
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
    # Calcula valor em USD por item usando a cotação já presente no fat_pedido_total
    .withColumn(
        "preco_usd",
        F.when(
            F.col("valor_total_pago_brl") > 0,
            F.col("preco_BRL") * (F.col("valor_total_pago_usd") / F.col("valor_total_pago_brl"))
        ).otherwise(F.lit(0))
    )
)

# Agrega por Ano, Mês e Categoria
df_fat_vendas_comercial = (
    df_vendas
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_produto").alias("qtd_itens_vendidos"),
        F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
        F.round(F.sum("preco_usd"), 2).alias("receita_total_usd"),
        F.round(F.sum("preco_BRL") / F.countDistinct("id_pedido"), 2).alias("ticket_medio_brl"),
    )
    .orderBy("ano_venda", "mes_venda")
)

(
    df_fat_vendas_comercial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.gold.fat_vendas_comercial")
)

print(f"✅ olist.gold.fat_vendas_comercial ({df_fat_vendas_comercial.count()} registros)")

In [0]:
# Enriquece produtos com categoria traduzida
df_produtos_com_categoria = (
    df_dim_produtos
    .join(df_dim_categoria, df_dim_produtos["categoria_produto"] == df_dim_categoria["nome_produto_pt"], "left")
    .select(
        df_dim_produtos["id_produto"],
        F.coalesce(df_dim_categoria["nome_produto_en"], df_dim_produtos["categoria_produto"]).alias("categoria_produto_en")
    )
)

# Agrega quantidade vendida por produto
df_ranking_produtos = (
    df_fat_itens_pedidos
    .join(df_produtos_com_categoria, "id_produto", "left")
    .groupBy("id_produto", "categoria_produto_en")
    .agg(F.count("id_produto").alias("quantidade_vendida"))
    .select(
        F.col("id_produto").alias("nome_produto"),
        F.col("categoria_produto_en").alias("categoria_produto"),
        "quantidade_vendida"
    )
)

df_top5_mais = (
    df_ranking_produtos
    .orderBy(F.col("quantidade_vendida").desc())
    .limit(5)
)

print("🏆 Top 5 Produtos MAIS Vendidos:")
display(df_top5_mais)

df_top5_menos = (
    df_ranking_produtos
    .orderBy(F.col("quantidade_vendida").asc())
    .limit(5)
)

print("📉 Top 5 Produtos MENOS Vendidos:")
display(df_top5_menos)